In [ ]:
%pip install pandas matplotlib seaborn entsoe-py matplotx # Installiert Phyton Bibliotheken (pandas matplotlib und seaborn) LK 11.06.2024

In [ ]:
import os                                   #importiert die os Bibliothek, um Betriebssystemfunktionen zu nutzen Lk 11.06.2024
import pandas as pd                         #importiert die pandas Bibliothek, um Daten zu analysieren und zu manipulieren Lk 11.06.2024
import matplotlib.pyplot as plt             #importiert die matplotlib Bibliothek, um Daten zu visualisieren Lk 11.06.2024
import matplotlib.dates as mdates           #importiert die mdates Funktion aus der matplotlib Bibliothek, um Datumsangaben zu formatieren Lk 11.06.2024
import seaborn as sns                       #importiert die seaborn Bibliothek, um ansprechende statistische Grafiken zu erstellen Lk 11.06.2024   
from entsoe import EntsoePandasClient       #importiert die EntsoePandasClient Klasse aus der entsoe Bibliothek, um Daten von der ENTSO-E API abzurufen Lk 11.06.2024
from concurrent.futures import ThreadPoolExecutor, as_completed # importiert Funktionen aus der concurrent.futures Bibliothek, um parallele Verarbeitung zu ermöglichen Lk 11.06.2024

In [ ]:
# ===== KONFIGURATION UND HILFSFUNKTIONEN =====

# API-Token (direkter Zugriff auf ENTSO-E API)
API_TOKEN = "9af21e20-861e-41a0-873b-03c15cddf7c9"  #mein Api token für die ENTSO-E API, um Daten abzurufen Lk 11.06.2024 , Leider nicht wirklich sicher - in einem echten Projekt würde ich diesen Token nicht direkt im Code speichern, sondern z.B. in einer Umgebungsvariable oder einem sicheren Vault.

# ENTSO-E Client initialisieren
client = EntsoePandasClient(api_key=API_TOKEN) #erstellt eine Instanz des EntsoePandasClient mit dem API-Token, um später Daten von der ENTSO-E API abzurufen Lk 11.06.2024

# Landes-Farbschema (einheitliche Farben für alle Diagramme)
LAND_FARBEN = {
    "Deutschland": "#1f77b4",
    "Frankreich": "#ff7f0e",
    "Spanien": "#2ca02c",
    "Italien": "#d62728",
    "Polen": "#9467bd",
    "Norwegen": "#8c564b",
    "Kroatien": "#e377c2",
    "Belgien": "#7f7f7f",
    "Niederlande": "#bcbd22",
    "Schweden": "#17becf",
    "Österreich": "#1f77b4",
    "Tschechien": "#ff7f0e",
}

def diagramm_pfad(ordner, dateiname):   #erstellt einen Ordner falls nötig und gibt den vollständigen Pfad zurück Lk 11.06.2024
    """Erstelle Ordner falls nötig und gebe vollständigen Pfad zurück"""
    import os
    os.makedirs(ordner, exist_ok=True)
    return os.path.join(ordner, dateiname)

def landfarbe(land): #setzt die Farbe für ein Land fest, damit alle Diagramme einheitlich aussehen Lk 11.06.2024
    """Gebe einheitliche Farbe für ein Land zurück"""
    return LAND_FARBEN.get(land, "#cccccc")

def landfarben(laender_liste): #setzt die Farben für eine Liste von Ländern fest, damit alle Diagramme einheitlich aussehen Lk 11.06.2024
    """Gebe Farbenliste für mehrere Länder zurück (für plt.plot() oder DataFrame.plot())"""
    return [landfarbe(land) for land in laender_liste]

# ===== STANDARDDEFINITIONEN =====
# Diese Variablen sind später mit Daten gefüllt

# Länder-Definitionen (wird in verschiedenen Zellen überschrieben) Lk 11.06.2024 - hier werden die Länder und ihre ENTSO-E Kürzel definiert, damit wir später einfach darauf zugreifen können
laender = {
    "Deutschland": "DE", "Frankreich": "FR", "Spanien": "ES",
    "Italien": "IT", "Polen": "PL", "Norwegen": "NO", "Kroatien": "HR"
}

# Placeholder für Jahresdaten
jahresdaten = {}
erzeugung_2025 = {}

# ===== ZUSÄTZLICHE HILFSFUNKTIONEN =====
# HINWEIS: hole_load_daten, hole_generation_daten, berechne_energie_mwh 
# sind in Zelle 2 definiert

def lade_aktueller_monat(client, laender): 
    """Lade Stromlastdaten für den aktuellen Monat aller Länder"""
    jetzt = pd.Timestamp.now(tz="UTC")                  #erstellt einen Zeitstempel für den aktuellen Zeitpunkt in UTC, um die Daten für den aktuellen Monat abzurufen Lk 11.06.2024
    monats_anfang = jetzt.replace(day=1)                #setzt den Tag des Zeitstempels auf 1, um den Beginn des Monats zu markieren, damit wir die Daten für den gesamten Monat abrufen können Lk 11.06.2024
    monatsdaten = {}                                    #erstellt ein leeres Dictionary, um die Daten für jedes Land zu speichern Lk 11.06.2024
    
    for land, kuerzel in laender.items():               #durchläuft jedes Land und sein Kürzel in der Länder-Definition, um die Daten für jedes Land abzurufen Lk 11.06.2024
        try:                                            #versucht, die Daten für das aktuelle Land abzurufen, und fängt Fehler ab, um sicherzustellen, dass der Prozess nicht unterbrochen wird, falls ein Land Probleme beim Laden der Daten hat Lk 11.06.2024
            daten = hole_load_daten(client, kuerzel, monats_anfang, jetzt)          #ruft die Funktion hole_load_daten auf, um die Stromlastdaten für das aktuelle Land und den aktuellen Monat abzurufen Lk 11.06.2024
            if hasattr(daten, 'squeeze'):               #wenn die daten eine 'squeeze'-Methode haben (z.B. wenn es sich um einen DataFrame mit nur einer Spalte handelt), wird diese Methode aufgerufen, um die Daten in eine einfachere Form zu bringen Lk 11.06.2024
                daten = daten.squeeze()                 #squeeze reduziert die Daten auf eine Serie, wenn es nur eine Spalte gibt, um die Handhabung zu erleichtern Lk 11.06.2024
            monatsdaten[land] = daten                   #speichert die abgerufenen Daten im monatsdaten Dictionary unter dem Namen des Landes, damit wir später alle Daten zusammenführen können Lk 11.06.2024
        except Exception as e:                          #wenn das land einen Fehler beim Laden der Daten verursacht, wird dieser abgefangen und eine Fehlermeldung mit dem Namen des Landes und der Fehlermeldung ausgegeben, um zu informieren, dass es ein Problem gab, aber der Prozess fortgesetzt wird Lk 11.06.2024
            print(f"{land}: Fehler beim Laden – {e}")   #wenn fehler, dann ausgabe des landes und der fehlermeldung, damit wir wissen, welches land betroffen ist und was das problem war Lk 11.06.2024
    
    if monatsdaten:                                     #wenn das monatsdaten Dictionary nicht leer ist (d.h. wenn mindestens ein Land erfolgreich Daten geladen hat), werden die Daten aller Länder mit pd.concat zu einem einzigen DataFrame zusammengeführt, damit wir alle Daten in einer Tabelle haben und sie leichter analysieren können Lk 11.06.2024
        return pd.concat(monatsdaten, axis=1)           #concat fügt alle Daten im monatsdaten Dictionary entlang der Spaltenachse (axis=1) zusammen, damit jedes Land eine eigene Spalte im resultierenden DataFrame hat Lk 11.06.2024
    else:   
        return pd.DataFrame()                       #wenn kein land erfolgreich daten geladen hat, wird ein leerer DataFrame zurückgegeben, damit wir eine konsistente Rückgabe haben, auch wenn es Probleme beim Laden der Daten gab Lk 11.06.2024

def lade_generation_woche(client, laender):             #Funktion zum laden der wochen generation jedes landes, damit wir die erzeugungsdaten für die aktuelle woche analysieren können Lk 11.06.2024
    """Lade Stromerzeugungsdaten für die aktuelle Woche aller Länder""" #Docstring, der erklärt, dass diese Funktion die Stromerzeugungsdaten für die aktuelle Woche für alle Länder lädt Lk 11.06.2024
    jetzt = pd.Timestamp.now(tz="UTC")                  #gleiches wie oben - erstellt einen Zeitstempel für den aktuellen Zeitpunkt in UTC, um die Daten für die aktuelle Woche abzurufen Lk 11.06.2024
    montag_diese_woche = jetzt.normalize() - pd.Timedelta(days=jetzt.weekday()) #setzt den montag dieser woche fest, indem es den aktuellen Zeitstempel auf den Beginn des Tages normalisiert und dann die Anzahl der Tage seit Montag subtrahiert, um den Montag dieser Woche zu erhalten, damit wir die Daten für die gesamte Woche abrufen können Lk 11.06.2024
    wochenbeginn = montag_diese_woche - pd.Timedelta(days=7)    #wochenbeginn ist der montag der vorwoche -> montag minus 7 tagen, damit wir die Daten für die gesamte Woche abrufen können Lk 11.06.2024
    wochenende = montag_diese_woche                             #wochenende ist der montag dieser woche, damit wir die Daten bis zum Ende der Woche abrufen können Lk 11.06.2024
    
    wochendaten = {}                                    #dictionary, für jedes land vergleich aktueller monat, damit wir die Daten für jedes Land speichern können Lk 11.06.2024
    for land, kuerzel in laender.items():
        try:
            daten = hole_generation_daten(client, kuerzel, wochenbeginn, wochenende)
            if hasattr(daten, 'squeeze'):
                daten = daten.squeeze()
            wochendaten[land] = daten
        except Exception as e:
            print(f"{land}: Fehler beim Laden – {e}")
    
    if wochendaten:
        return pd.concat(wochendaten, axis=1)
    else:
        return pd.DataFrame()

print("✅ Setup abgeschlossen: API ist konfiguriert, Daten können jetzt heruntergeladen werden!") #wenn alles geklappt hat das ausgeben

In [ ]:
# Festplatten-Cache: Einmal geladene Daten werden als Datei im Ordner "daten_cache"
# gespeichert. Beim nächsten Durchlauf (auch nach einem Kernel-Neustart!) kommen die
# Daten von der Festplatte statt von der API - das spart Minuten an Wartezeit.
# Zum Aktualisieren der Daten einfach den Ordner "daten_cache" löschen.
CACHE_DIR = "daten_cache"           #festlegen namen und pfad für den cache ordner, damit wir die daten lokal speichern können und nicht jedes mal von der api laden müssen Lk 11.06.2024
os.makedirs(CACHE_DIR, exist_ok=True)

def hole_lastdaten(client, art, kuerzel, start_iso, end_iso):                               #funktion die daten von der api holt und im cache speichert, damit wir die daten nicht jedes mal von der api laden müssen und schneller arbeiten können Lk 11.06.2024
    """Allgemeine Funktion zum Abrufen von Stromdaten von der ENTSO-E API (mit Cache)"""
    # Dateiname aus den Abfrage-Parametern bauen, z.B. "load_DE_2023-01-01_2024-01-01.pkl"
    datei = os.path.join(CACHE_DIR, f"{art}_{kuerzel}_{start_iso[:10]}_{end_iso[:10]}.pkl") #baut wie die datei ausshen soll die heruntergeladen wird, damit wir die daten lokal speichern können und nicht jedes mal von der api laden müssen Lk 11.06.2024

    # Schon im Cache? Dann direkt von der Festplatte laden (dauert Millisekunden)
    if os.path.exists(datei):
        daten = pd.read_pickle(datei) #was ist pickeld?:_ - pd.read_pickle liest die Daten aus der Cache-Datei, die mit pd.to_pickle gespeichert wurde, und lädt sie als DataFrame zurück, damit wir die Daten schnell laden können, ohne sie jedes Mal von der API abrufen zu müssen Lk 11.06.2024
    else:
        # Nicht im Cache: von der API holen (dauert je nach Zeitraum einige Sekunden)
        start = pd.Timestamp(start_iso) #konvertiert den Startzeitpunkt von einem ISO-String in einen pandas Timestamp, damit wir ihn für die API-Abfrage verwenden können Lk 11.06.2024
        end = pd.Timestamp(end_iso)
        if art == "load":                   #wenn die art "load" ist, dann werden die lastdaten von der api abgefragt, damit wir die stromlastdaten für das land und den zeitraum erhalten können Lk 11.06.2024
            daten = client.query_load(kuerzel, start=start, end=end)  # Stromlast in MW 
        elif art == "generation":           #wenn die art "generation" ist, dann werden die erzeugungsdaten von der api abgefragt, damit wir die stromerzeugungsdaten für das land und den zeitraum erhalten können Lk 11.06.2024
            daten = client.query_generation(kuerzel, start=start, end=end)  # Stromerzeugung in MW
        else:
            raise ValueError(f"Unbekannte Abfrageart: {art}") #sonst wenn art weder "load" noch "generation" ist, wird eine Fehlermeldung ausgegeben, um zu informieren, dass die Abfrageart nicht erkannt wurde Lk 11.06.2024

        # Erst in eine temporäre Datei schreiben, dann umbenennen -
        # so entstehen keine kaputten Cache-Dateien, falls der Download abbricht
        daten.to_pickle(datei + ".tmp")
        os.replace(datei + ".tmp", datei)

    # WICHTIG: Auf UTC vereinheitlichen! ENTSO-E liefert jedes Land in seiner lokalen
    # Zeitzone (Europe/Berlin, Europe/Paris, ...). Beim Zusammenfügen mehrerer Länder
    # (pd.concat) würde pandas sonst die Monatsgrenzen verschieben - dadurch ging
    # vorher der Januar verloren und der "Dezember" war fast leer.
    return daten.tz_convert("UTC") #

# Vereinfachte Wrapper-Funktionen für Last- und Erzeugungsdaten
def hole_load_daten(client, kuerzel, start, end):
    """Ruft Lastdaten ab - konvertiert Timestamps zu ISO-Strings für den Cache-Dateinamen"""
    return hole_lastdaten(client, "load", kuerzel, start.isoformat(), end.isoformat())

def hole_generation_daten(client, kuerzel, start, end):
    """Ruft Erzeugungsdaten ab - konvertiert Timestamps zu ISO-Strings für den Cache-Dateinamen"""
    return hole_lastdaten(client, "generation", kuerzel, start.isoformat(), end.isoformat())

def berechne_energie_mwh(daten):
    """Rechnet Leistung (MW) in Energie (MWh) um.

    WICHTIG: Jeder Datenpunkt wird mit seinem ECHTEN Zeitintervall gewichtet,
    denn manche Länder stellen mitten im Jahr die Meldeauflösung um
    (Polen 2024, Norwegen 2025: von 60 auf 15 Minuten). Mit einem festen
    Intervall wären die Energiewerte dann bis zu 4x zu hoch.
    """
    # Abstand zum jeweils NÄCHSTEN Zeitstempel in Stunden (= Geltungsdauer des Werts)
    dauer_h = daten.index.to_series().diff().shift(-1).dt.total_seconds() / 3600  ## Berechnet die Geltungsdauer jedes Leistungswerts anhand des Abstands zum nächsten Zeitstempel in Stunden.Zurondung des vorrangehenden zeitpunkts -> das bei startzeit nicht 0 steht, sondern die tatsächliche Dauer bis zum nächsten Zeitstempel, damit wir die Energie korrekt berechnen können Lk 11.06.2024
    dauer_h = dauer_h.ffill()  # letzter Punkt: Intervall des vorletzten übernehmen #anders herum hier hat der letzt punkt keinen referenzwert also gehen wir vom vorherigen aus, damit wir auch für den letzten Datenpunkt eine gültige Dauer haben Lk 11.06.2024
    return daten.mul(dauer_h, axis=0) ## Multipliziert jeden Leistungswert mit seiner zugehörigen Dauer und berechnet so die Energie in MWh.


In [ ]:
# TURBO-ZELLE: Lädt alle Daten, die das Notebook braucht, PARALLEL in den Cache.
# Die API-Abfragen sind reine Wartezeit - mit mehreren gleichzeitigen Abfragen ist
# der erste Durchlauf ca. 4-5x schneller. Ab dem zweiten Durchlauf kommt sowieso
# alles aus dem Festplatten-Cache und diese Zelle ist in Sekunden fertig.
from concurrent.futures import ThreadPoolExecutor, as_completed

def vorladen(max_worker=5): #definiert funktion vorladen, die bis zu 'max_worker' Abfragen gleichzeitig startet, um die Daten für das Notebook schneller in den Cache zu laden Lk 11.06.2024
    jetzt = pd.Timestamp.now(tz="UTC")
    # Gleiche Zeitraum-Berechnung wie in den Wochen-/Monats-Zellen weiter unten
    montag_diese_woche = jetzt.normalize() - pd.Timedelta(days=jetzt.weekday())
    wochenbeginn = montag_diese_woche - pd.Timedelta(days=7)
    monatsbeginn_aktuell = jetzt.normalize().replace(day=1)
    monatsbeginn = (monatsbeginn_aktuell - pd.Timedelta(days=1)).replace(day=1)

    # Liste aller Abfragen, die im Notebook vorkommen
    jobs = []
    for jahr in [2023, 2024, 2025]:  # komplette Jahre: Last + Erzeugung für alle Länder LK 11.06.2024
        s = pd.Timestamp(f"{jahr}-01-01", tz="UTC")
        e = pd.Timestamp(f"{jahr + 1}-01-01", tz="UTC")
        for kuerzel in laender.values():
            jobs.append(("load", kuerzel, s, e))
            jobs.append(("generation", kuerzel, s, e))
    for kuerzel in laender.values():  # vorherige Woche + vorheriger Monat
        jobs.append(("load", kuerzel, wochenbeginn, montag_diese_woche))
        jobs.append(("generation", kuerzel, wochenbeginn, montag_diese_woche))
        jobs.append(("load", kuerzel, monatsbeginn, monatsbeginn_aktuell))

    def lade_einen(job):    #funktion, die einen einzelnen job ausführt, damit wir die daten für jede abfrage parallel laden können Lk 11.06.2024
        art, kuerzel, s, e = job
        # Eigener Client pro Abfrage, weil die HTTP-Verbindung nicht thread-sicher ist
        eigener_client = EntsoePandasClient(api_key=API_TOKEN)
        hole_lastdaten(eigener_client, art, kuerzel, s.isoformat(), e.isoformat())

    # Bis zu 'max_worker' Abfragen gleichzeitig starten (ENTSO-E erlaubt 400/Minute)
    with ThreadPoolExecutor(max_workers=max_worker) as pool:
        futures = {pool.submit(lade_einen, job): job for job in jobs}
        fertig = 0
        for future in as_completed(futures):
            art, kuerzel, _, _ = futures[future]
            fertig += 1
            try:
                future.result()
                print(f"[{fertig}/{len(jobs)}] {kuerzel} {art} ✅")
            except Exception as e:
                print(f"[{fertig}/{len(jobs)}] {kuerzel} {art} ❌ {e}")

vorladen()


In [ ]:
# Monats- und Jahreslast für alle benötigten Jahre berechnen
def lade_monatliche_last(jahr):
    start = pd.Timestamp(f"{jahr}-01-01", tz="UTC")
    ende = pd.Timestamp(f"{jahr + 1}-01-01", tz="UTC")
    monatsdaten = {}

    for land, kuerzel in laender.items():
        try:
            last_mw = hole_load_daten(client, kuerzel, start, ende).squeeze()
            energie_mwh = berechne_energie_mwh(last_mw)
            monatsdaten[land] = energie_mwh.resample("MS").sum() / 1_000_000
        except Exception as e:
            print(f"{land} ({jahr}): Fehler beim Laden - {e}")

    return pd.DataFrame(monatsdaten)


df_2023 = lade_monatliche_last(2023)
df_2024 = lade_monatliche_last(2024)
df_2025 = lade_monatliche_last(2025)

# Summe der zwölf Monatswerte je Land
jahresdaten = df_2023.sum(axis=0, min_count=1).dropna().to_dict()
df = pd.DataFrame.from_dict(jahresdaten, orient="index", columns=["TWh"])
df = df.sort_values("TWh", ascending=False)

if df.empty:
    print("Keine gültigen Jahresdaten für 2023 gefunden.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    balken = ax.barh(df.index[::-1], df["TWh"][::-1], color="#4C72B0", edgecolor="white")
    ax.bar_label(balken, fmt="%.0f", padding=3, fontsize=9)

    ax.set_title("Jährliche Last im europäischen Vergleich (2023)", fontsize=14)
    ax.set_xlabel("Aggregierte Last in TWh")
    ax.set_ylabel("Land")
    ax.grid(axis="x", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(diagramm_pfad("Diagramme_2023", "lastvergleich.png"), dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# Erstelle Dictionary mit Jahr als Schlüssel und Daten als Wert
jahre_dict = {"2023": df_2023, "2024": df_2024, "2025": df_2025}

# Schleife: Erstelle für jedes Jahr einen separaten Linienplot
for jahr, df in jahre_dict.items():  # jahr = "2023" usw., df = die zugehörigen Daten
    # Erstelle neue Figur und Achse für das Diagramm
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Zeichne für jedes Land eine eigene Linie
    for land in df.columns:  # Schleife über alle Ländernamen (Spaltennamen)
        ax.plot(df.index, df[land], marker="o", label=land, color=landfarbe(land))
    
    # Beschriftungen
    ax.set_title(f"Monatliche Last im europäischen Vergleich ({jahr})", fontsize=14)
    ax.set_xlabel("Monat")
    ax.set_ylabel("Last in TWh")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))  # Legende rechts NEBEN dem Diagramm
    ax.grid(linestyle="--", alpha=0.5)  # Gitternetz im Hintergrund
    ax.tick_params(axis='x', rotation=45)  # Drehe X-Achsen-Labels um 45°
    
    plt.tight_layout()  # Passe Abstände an
    plt.savefig(diagramm_pfad(f"Diagramme_{jahr}", "lastgang.png"), bbox_inches="tight")  # Speichere im Jahr-Ordner
    plt.close()  # Schließe Figur (gibt Speicher frei)

print("✅ Linienplots für 2023, 2024, 2025 in separaten Ordnern abgespeichert")

In [ ]:
# Funktion: Lade VORHERIGE Woche (komplette Woche: Montag-Sonntag)
def lade_aktuelle_woche(client, laender):
    """
    Lädt den stündlichen Lastverlauf der VORHERIGEN WOCHE für alle Länder.
    Immer vollständig: von Montag bis Sonntag (7 Tage komplett)
    """
    # Hole aktuelle Zeit
    jetzt = pd.Timestamp.now(tz="UTC")
    
    # Berechne Montag dieser Woche
    montag_diese_woche = jetzt.normalize() - pd.Timedelta(days=jetzt.weekday())
    
    # Berechne Montag und Sonntag der VORHERIGEN Woche
    wochenbeginn = montag_diese_woche - pd.Timedelta(days=7)  # Montag der Woche davor
    wochenende = wochenbeginn + pd.Timedelta(days=7)  # Montag der aktuellen Woche (= Sonntag+1)

    wochendaten = {}
    for land, kuerzel in laender.items():
        try:
            # Hole stündliche Lastdaten für die vorherige komplette Woche
            daten = hole_load_daten(client, kuerzel, wochenbeginn, wochenende)
            # Falls Daten ein DataFrame sind, konvertiere zu Series (eindimensional)
            if hasattr(daten, 'squeeze'):
                daten = daten.squeeze()
            wochendaten[land] = daten
            print(f"{land}: Daten geladen ✅")
        except Exception as e:
            print(f"{land}: Fehler – {e}")

    # Kombiniere alle Länder zu einem DataFrame (Länder = Spalten, Zeit = Zeilen)
    if not wochendaten:
        return pd.DataFrame()

    return pd.concat(wochendaten, axis=1)

In [ ]:
# Wochendaten laden, bevor sie im Diagramm verwendet werden
print("=== Last vorherige vollständige Woche ===")
df_woche = lade_aktuelle_woche(client, laender)

if df_woche.empty:
    print("Keine Wochendaten zum Zeichnen vorhanden.")
else:
    fig, ax = plt.subplots(figsize=(16, 6))

    for land in df_woche.columns:
        ax.plot(df_woche.index, df_woche[land], label=land, color=landfarbe(land))

    ax.set_title("Lastverlauf - vorherige vollständige Woche (ENTSO-E: Actual Total Load)", fontsize=13)
    ax.set_xlabel("Datum / Uhrzeit")
    ax.set_ylabel("Last in MW")
    ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1))
    ax.grid(linestyle="--", alpha=0.5)

    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m."))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(
        diagramm_pfad("Diagramme_aktuell", "lastverlauf_woche.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
print("=== Aktueller Monat ===")
# Lade Lastdaten für den aktuellen Monat (von Zelle weiter oben)
df_monat_aktuell = lade_aktueller_monat(client, laender)

# Erstelle Liniendiagramm für den aktuellen Monat mit Uhrzeiten
fig, ax = plt.subplots(figsize=(18, 6))

# Zeichne für jedes Land eine Linie
for land in df_monat_aktuell.columns:
    ax.plot(df_monat_aktuell.index, df_monat_aktuell[land], label=land, color=landfarbe(land))

# Beschriftungen
ax.set_title("Aktueller Lastverlauf – dieser Monat (ENTSO-E: Actual Total Load in MW)", fontsize=13)
ax.set_xlabel("Datum / Uhrzeit")
ax.set_ylabel("Last in MW")
ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1))  # Legende rechts neben dem Diagramm
ax.grid(linestyle="--", alpha=0.5)

# Automatische Tick-Platzierung für Datums-/Uhrzeitleisten
ax.xaxis.set_major_locator(mdates.DayLocator())  # Hauptticks: jeden Tag
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m. %H:%M"))  # Format: Datum + Uhrzeit
ax.xaxis.set_minor_locator(mdates.HourLocator(interval=12))  # Nebenticks: alle 12 Stunden

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(diagramm_pfad("Diagramme_aktuell", "lastverlauf_monat.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Auch hier: Tages-Energien statt 20.000 unsichtbarer Viertelstunden-Balken
tagesenergie_monat = berechne_energie_mwh(df_monat_aktuell).resample("D").sum() / 1000  # GWh
tagesenergie_monat.index = tagesenergie_monat.index.strftime("%d.%m.")

fig, ax = plt.subplots(figsize=(18, 6))
tagesenergie_monat.plot(kind="bar", ax=ax, color=landfarben(tagesenergie_monat.columns),
                        edgecolor="white", width=0.8)

# Beschriftungen
ax.set_title("Tagesenergie – vorheriger Monat (aus ENTSO-E: Actual Total Load)", fontsize=13)
ax.set_xlabel("Tag")
ax.set_ylabel("Energie in GWh pro Tag")
ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1), fontsize=9)  # Legende rechts neben dem Diagramm
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(diagramm_pfad("Diagramme_aktuell", "lastverlauf_balken_monat.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
print("\n=== Generation aktuelle Woche ===")
df_gen_woche = lade_generation_woche(client, laender)

In [ ]:
# Funktion: Berechne Anteil erneuerbarer Energien
def lade_erneuerbare(client, laender, jahr):
    """
    Berechnet für jedes Land: % der Erzeugung aus erneuerbaren Energien
    """
    # Definiere Zeitraum für das ganze Jahr
    start = pd.Timestamp(f"{jahr}-01-01", tz="UTC")
    end   = pd.Timestamp(f"{jahr+1}-01-01", tz="UTC")

    # Liste der erneuerbaren Energietypen (wie ENTSO-E sie nennt)
    erneuerbare_typen = [
        "Solar", "Wind Onshore", "Wind Offshore",
        "Hydro Run-of-river and poundage", "Hydro Water Reservoir",
        "Geothermal", "Biomass"
    ]

    ergebnis = {}
    for land, kuerzel in laender.items():
        try:
            # Hole Erzeugungsdaten (alle Energiearten)
            daten = hole_generation_daten(client, kuerzel, start, end)
            # Vereinfache Multi-Index Spalten (pandas-technisch)
            if isinstance(daten.columns, pd.MultiIndex):
                daten.columns = daten.columns.get_level_values(0)
            
            # In Energie umrechnen (jeder Datenpunkt mit echtem Zeitintervall gewichtet)
            energie = berechne_energie_mwh(daten)
            # Berechne Gesamterzeugung (Summe aller Energiearten)
            gesamt = energie.sum(axis=1).sum()  # axis=1 = über Energiearten, dann über Zeit
            # Berechne nur Erzeugung aus erneuerbaren (die in der Liste sind)
            erneuerbar = energie[[s for s in erneuerbare_typen if s in energie.columns]].sum(axis=1).sum()
            # Berechne Prozentanteil: (erneuerbar / gesamt) * 100
            ergebnis[land] = round((erneuerbar / gesamt) * 100, 2)
            print(f"{land}: {ergebnis[land]}% erneuerbar")
        except Exception as e:
            print(f"{land}: Fehler – {e}")

    # Gebe Ergebnis als pandas Series zurück (sortiert absteigend)
    return pd.Series(ergebnis).sort_values(ascending=False)

# Lade und visualisiere Erneuerbare für 2023
print("=== Anteil Erneuerbare 2023 ===")
erneuerbare_2023 = lade_erneuerbare(client, laender, 2023)

# Erstelle horizontales Balkendiagramm
fig, ax = plt.subplots(figsize=(10, 6))
# Färbe grün (über 50%) oder blau (unter 50%) - gedämpfte Farbtöne wirken moderner
farben = ["#55A868" if v >= 50 else "#4C72B0" for v in erneuerbare_2023.values]
balken = ax.barh(erneuerbare_2023.index[::-1], erneuerbare_2023.values[::-1],
                 color=farben[::-1], edgecolor="white")
ax.bar_label(balken, fmt="%.1f%%", padding=3, fontsize=9)  # Wert direkt am Balken
ax.axvline(50, color="grey", linestyle="--", alpha=0.8, label="50% Schwelle")  # Referenzlinie
ax.set_title("Anteil erneuerbarer Energien an der Erzeugung (2023)", fontsize=13)
ax.set_xlabel("Anteil in %")
ax.set_ylabel("Land")
ax.legend(loc="lower right")  # Unten rechts ist bei absteigender Sortierung frei
ax.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(diagramm_pfad("Diagramme_2023", "erneuerbare_anteil_2023.png"))
plt.show()

In [ ]:
# Importiere seaborn - Bibliothek für schönere statistische Plots
import seaborn as sns

# Erstelle eine Heatmap (Wärmekarte) für 2023
fig, ax = plt.subplots(figsize=(12, 5))

# Transponiere df_2023: Länder als Zeilen, Monate als Spalten (bessere Sichtbarkeit)
sns.heatmap(
    df_2023.T,  # T = Transpose (umdrehen/wechseln)
    annot=True,  # annot=True: Zeige Zahlenwerte in den Zellen
    fmt=".1f",  # Formatiere auf 1 Dezimalstelle
    cmap="YlOrRd",  # Farbschema: Gelb (niedrig) bis Rot (hoch)
    linewidths=0.5,  # Dünne Linien zwischen den Zellen
    ax=ax,  # Zeichne auf unsere Achse
    cbar_kws={"label": "Last in TWh"}  # Beschriftung der Farblegende
)

# Beschriftungen
ax.set_title("Heatmap: Monatliche Last pro Land (2023)", fontsize=13)
ax.set_xlabel("Monat")  # Monats-Nummern (01, 02, ... 12)
ax.set_ylabel("Land")  # Ländernamen
plt.tight_layout()
plt.savefig(diagramm_pfad("Diagramme_2023", "heatmap.png"))
plt.show()

In [ ]:
print("=== Anteil Erneuerbare ===")

# Schleife: Erstelle für jedes Jahr (2023, 2024, 2025) ein separates Diagramm
for jahr in [2023, 2024, 2025]:
    print(f"\nGeneriere Diagramm für {jahr}...")
    # Berechne Erneuerbaren-Anteil für dieses Jahr
    erneuerbare = lade_erneuerbare(client, laender, jahr)
    
    # Erstelle horizontales Balkendiagramm (Ländernamen lesbar, Werte am Balken)
    fig, ax = plt.subplots(figsize=(10, 6))
    balken = ax.barh(erneuerbare.index[::-1], erneuerbare.values[::-1],
                     color="#4C72B0", edgecolor="white")
    ax.bar_label(balken, fmt="%.1f%%", padding=3, fontsize=9)
    # Zeichne graue Referenzlinie bei 50%
    ax.axvline(50, color="grey", linestyle="--", alpha=0.8, linewidth=2, label="50% Schwelle")
    ax.set_title(f"Anteil erneuerbarer Energien an der Erzeugung ({jahr})", fontsize=13)
    ax.set_xlabel("Anteil in %")
    ax.set_ylabel("Land")
    ax.legend(loc="lower right")  # Unten rechts ist bei absteigender Sortierung frei
    ax.grid(axis="x", linestyle="--", alpha=0.5)
    ax.set_xlim(0, 115)  # Platz für die Wertelabels lassen
    plt.tight_layout()
    # Speichere im Jahr-Ordner
    plt.savefig(diagramm_pfad(f"Diagramme_{jahr}", "erneuerbare_anteil.png"))
    plt.close()  # Gib Speicher frei

print("✅ Erneuerbare-Anteile für 2023, 2024, 2025 in separaten Ordnern abgespeichert")

In [ ]:
# Importiere seaborn für Heatmaps (Wärmekarten)
import seaborn as sns

# Dictionary mit den drei Jahren
jahre_dict = {"2023": df_2023, "2024": df_2024, "2025": df_2025}

# Schleife: Erstelle für jedes Jahr eine separate Heatmap
for jahr, df in jahre_dict.items():
    # Erstelle neue Figur für Heatmap
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Erstelle Heatmap: Transponiere die Daten (Länder als Zeilen, Monate als Spalten)
    sns.heatmap(
        df.T,  # T = Transpose (umdrehen)
        annot=True,  # annot=True: Zeige Zahlen in den Zellen
        fmt=".1f",  # Formatiere Zahlen auf 1 Dezimalstelle
        cmap="YlOrRd",  # Farbschema: Gelb-Orange-Rot (hellgelb = niedrig, dunkelrot = hoch)
        linewidths=0.5,  # Zeichne dünne Linien zwischen Zellen
        ax=ax,  # Zeichne auf unsere Achse
        cbar_kws={"label": "Last in TWh"}  # Beschriftung der Farbbalken-Legende
    )
    
    # Beschriftungen
    ax.set_title(f"Heatmap: Monatliche Last pro Land ({jahr})", fontsize=13)
    ax.set_xlabel("Monat")  # X-Achse
    ax.set_ylabel("Land")  # Y-Achse
    
    plt.tight_layout()
    # Speichere im Jahr-Ordner
    plt.savefig(diagramm_pfad(f"Diagramme_{jahr}", "heatmap.png"))
    plt.close()  # Gib Speicher frei

print("✅ Heatmaps für 2023, 2024, 2025 in separaten Ordnern abgespeichert")

In [ ]:
# Farben für die Energieträger (passend zur Realität)
farben_traeger = {
    "Solar": "#FFD700",
    "Wind Onshore": "#87CEEB",
    "Wind Offshore": "#4682B4",
    "Hydro Run-of-river and poundage": "#1E90FF",
    "Hydro Water Reservoir": "#0000CD",
    "Hydro Pumped Storage": "#00008B",
    "Biomass": "#228B22",
    "Geothermal": "#8B4513",
    "Nuclear": "#FF6347",
    "Fossil Hard coal": "#2F4F4F",
    "Fossil Brown coal/Lignite": "#8B4513",
    "Fossil Gas": "#FF8C00",
    "Fossil Oil": "#000000",
    "Waste": "#A0522D",
    "Other": "#808080",
    "Other renewable": "#90EE90"
}

# Erstelle untereinander für jedes Land ein Diagramm
if not erzeugung_2025:
    print("⚠️ Keine Erzeugungsdaten für 2025 vorhanden — Diagramm übersprungen")
else:
    fig, axes = plt.subplots(len(erzeugung_2025), 1,
                             figsize=(14, 5 * len(erzeugung_2025)))

    # Falls nur ein Land: axes ist kein Array, sondern einzelnes Objekt
    if len(erzeugung_2025) == 1:
        axes = [axes]

    for ax, (land, df) in zip(axes, erzeugung_2025.items()):
        # Farben für vorhandene Energieträger zuweisen
        farben = [farben_traeger.get(col, "#CCCCCC") for col in df.columns]
        
        # Gestapeltes Balkendiagramm (zeigt jeden Energieträger separat)
        df.plot(kind="bar", stacked=True, ax=ax, color=farben, edgecolor="white", width=0.8)
        
        ax.set_title(f"{land} – Erzeugung pro Energieträger (2025)", fontsize=13)
        ax.set_xlabel("Monat")
        ax.set_ylabel("Erzeugung in TWh")
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        plt.setp(ax.get_xticklabels(), rotation=45)

    plt.suptitle("Energieträger-Mix pro Land (2025)", fontsize=15, y=1.001)
    plt.tight_layout()
    plt.savefig(diagramm_pfad("Diagramme_2025", "erzeugung_nach_traeger.png"), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# 1. Vorbereitung: Zeitraum und Länder definieren
start = pd.Timestamp("2025-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")

laender = {
    "Deutschland": "DE", "Frankreich": "FR", "Spanien": "ES", 
    "Italien": "IT", "Polen": "PL", "Norwegen": "NO", "Kroatien": "HR"
}

monatsdaten_wind = {}

print("Starte Datenabruf für das Jahr 2025 (kommt aus dem Cache, falls schon geladen)...")

# 2. Schleife: Daten für jedes Land einzeln abrufen und verarbeiten
for land, kuerzel in laender.items():
    print(f"Lade Daten für {land}...")
    try:
        # Komplette Erzeugungsdaten für 2025 holen
        gen_daten = hole_generation_daten(client, kuerzel, start, end)  # nutzt den Cache!
        
        # Eine leere Serie für den Windstrom anlegen
        wind_gesamt = pd.Series(dtype=float)
        
        # Wind Onshore (an Land) hinzufügen, falls vorhanden
        if 'Wind Onshore' in gen_daten.columns:
            wind_on = gen_daten['Wind Onshore']
            # Sicherheitscheck gegen doppelte Spalten (wie vorhin bei Spanien)
            if isinstance(wind_on, pd.DataFrame): wind_on = wind_on.sum(axis=1)
            wind_gesamt = wind_gesamt.add(wind_on, fill_value=0)
            
        # Wind Offshore (auf See) hinzufügen, falls vorhanden
        if 'Wind Offshore' in gen_daten.columns:
            wind_off = gen_daten['Wind Offshore']
            if isinstance(wind_off, pd.DataFrame): wind_off = wind_off.sum(axis=1)
            wind_gesamt = wind_gesamt.add(wind_off, fill_value=0)

        # Wenn wir Daten gefunden haben, rechnen wir sie in monatliche Energie um
        if not wind_gesamt.empty:
            # Leistung (MW) × echtes Zeitintervall (h) = Energie (MWh)
            energie = berechne_energie_mwh(wind_gesamt)

            # Alle Werte nach dem jeweiligen Monat (1 bis 12) gruppieren und summieren
            monatssummen = energie.groupby(energie.index.month).sum()
            
            # In Terawattstunden (TWh) umrechnen für kleinere, lesbare Zahlen im Diagramm
            monatsdaten_wind[land] = monatssummen / 1_000_000
            
    except Exception as e:
        print(f"  -> Konnte {land} nicht komplett verarbeiten (Fehler: {e})")

# 3. Tabelle aus den Ergebnissen bauen
wind_tabelle = pd.DataFrame(monatsdaten_wind)

# Monatszahlen (1-12) durch Namen ersetzen
monatsnamen = ['Jan', 'Feb', 'Mär', 'Apr', 'Mai', 'Jun', 'Jul', 'Aug', 'Sep', 'Okt', 'Nov', 'Dez']
wind_tabelle.index = monatsnamen

# 4. Das Liniendiagramm zeichnen
wind_tabelle.plot(
    kind='line',
    marker='o',       # Setzt kleine Punkte auf die Graphen für jeden Monat
    figsize=(12, 6),
    linewidth=2,
    color=landfarben(wind_tabelle.columns)  # Feste Länderfarben
)

plt.title("Monatliche Windstromerzeugung im Europavergleich (2025)", fontsize=14)
plt.ylabel("Erzeugte Energie in Terawattstunden (TWh)", fontsize=12)
plt.grid(axis='both', linestyle='--', alpha=0.5)

# Legende ordentlich neben das Diagramm setzen, damit nichts überdeckt wird
plt.legend(title="Land", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

plt.savefig(diagramm_pfad("Diagramme_2025", "wind_monatlich.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 1. Zeitraum und Land festlegen (Wir nehmen eine winterliche Woche in Deutschland)
start = pd.Timestamp("2024-01-15", tz="UTC")
end = pd.Timestamp("2024-01-22", tz="UTC")
land = "DE"

print(f"Lade Erzeugungsdaten für {land}...")
gen_daten = hole_generation_daten(client, land, start, end)  # nutzt den Cache!

# Sicherheitscheck für neuere Pandas-Versionen: 
# Tabelle kurz kippen (.T), doppelte Namen zusammenrechnen, zurückkippen (.T) und Lücken füllen
gen_daten = gen_daten.T.groupby(level=0).sum().T.fillna(0)

# 2. CO2-Emissionsfaktoren definieren (in Gramm CO2 pro kWh)
co2_faktoren = {
    'Biomass': 230,
    'Fossil Brown coal/Lignite': 1000, # Braunkohle
    'Fossil Gas': 400,                 # Erdgas
    'Fossil Hard coal': 800,           # Steinkohle
    'Fossil Oil': 800,                 # Öl
    'Nuclear': 0,
    'Solar': 0,
    'Waste': 500,                      # Müllverbrennung
    'Wind Offshore': 0,
    'Wind Onshore': 0,
    'Hydro Pumped Storage': 0,
    'Hydro Run-of-river and poundage': 0,
    'Hydro Water Reservoir': 0,
    'Geothermal': 0,
    'Other renewable': 0,
    'Other': 300                       # Schätzwert für Unbekanntes
}

# 3. Berechnung der CO2-Intensität
gesamt_co2 = pd.Series(0.0, index=gen_daten.index)
gesamt_strom = gen_daten.sum(axis=1) 

for kraftwerkstyp in gen_daten.columns:
    faktor = co2_faktoren.get(kraftwerkstyp, 0)
    gesamt_co2 += gen_daten[kraftwerkstyp] * faktor

# Intensität = Gesamtes CO2 geteilt durch gesamten Strom
co2_intensitaet = gesamt_co2 / gesamt_strom

# 4. Das Diagramm zeichnen
import matplotlib.pyplot as plt

co2_intensitaet.plot(
    kind='line',
    figsize=(12, 6),
    color='firebrick',
    linewidth=2
)

# Grüne Hilfslinie (Zielwert)
plt.axhline(y=100, color='green', linestyle='--', alpha=0.7, label='Klimaziel für sauberen Strom (100 g/kWh)')

plt.title("CO2-Intensität des deutschen Stromnetzes (15. - 22. Jan 2024)", fontsize=14)
plt.ylabel("Gramm CO2 pro produzierter Kilowattstunde (g/kWh)", fontsize=12)
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))  # Legende rechts neben dem Diagramm
plt.tight_layout()

plt.savefig(diagramm_pfad("Diagramme_vergleich", "co2_intensitaet_de_woche.png"), dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
# 1. Vorbereitung: Zeitraum und Länder
start = pd.Timestamp("2025-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")

laender = {
    "Deutschland": "DE", "Frankreich": "FR", "Spanien": "ES", 
    "Italien": "IT", "Norwegen": "NO", "Kroatien": "HR"
}

# NEU: Alle Werte sind jetzt offizielle IPCC-Lebenszyklus-Emissionen (Median-Werte in g CO2eq/kWh)
co2_faktoren = {
    'Fossil Brown coal/Lignite': 1050, # Braunkohle (Abbau & Verbrennung)
    'Fossil Hard coal': 820,           # Steinkohle
    'Fossil Oil': 840,                 # Öl
    'Fossil Gas': 490,                 # Gas (inkl. Förderung & Transport)
    'Waste': 500,                      # Müllverbrennung
    'Biomass': 230,                    # Biomasse (Anbau, Ernte, Transport)
    'Solar': 41,                       # Photovoltaik (Produktion der Module)
    'Geothermal': 38,                  # Geothermie
    'Hydro Pumped Storage': 24,        # Wasserkraft (Bau der Anlagen)
    'Hydro Run-of-river and poundage': 24, 
    'Hydro Water Reservoir': 24,
    'Wind Offshore': 12,               # Wind auf See
    'Nuclear': 12,                     # Kernkraft (Uranabbau, Bau, Endlager)
    'Wind Onshore': 11,                # Wind an Land
    'Other renewable': 30,             # Schätzwert für sonstige
    'Other': 300                       # Schätzwert für Unbekanntes
}

eu_co2_emissionen = pd.DataFrame()

print("Starte die Berechnung der globalen Lebenszyklus-Emissionen (2025)...")

# 2. Schleife: Alle Länder durchgehen
for land, kuerzel in laender.items():
    print(f"Berechne Daten für {land}...")
    try:
        gen_daten = hole_generation_daten(client, kuerzel, start, end)  # nutzt den Cache!
        # Sicherheitscheck: Doppelte Spalten zusammenfassen
        gen_daten = gen_daten.T.groupby(level=0).sum().T.fillna(0)
        
        if not gen_daten.empty:
            # MW → MWh mit echtem Zeitintervall pro Datenpunkt (wichtig für Norwegen 2025!)
            energie = berechne_energie_mwh(gen_daten)

            monats_energie = energie.groupby(energie.index.month).sum()
            monats_co2 = pd.DataFrame(index=monats_energie.index)
            
            for kraftwerk in monats_energie.columns:
                faktor = co2_faktoren.get(kraftwerk, 0)
                if faktor > 0: # Jetzt rutschen ALLE Energiequellen hier durch!
                    # (MWh * g/kWh) / 1.000.000 = Millionen Tonnen CO2 (Megatonnen)
                    monats_co2[kraftwerk] = (monats_energie[kraftwerk] * faktor) / 1_000_000
                    
            eu_co2_emissionen = eu_co2_emissionen.add(monats_co2, fill_value=0)
            
    except Exception as e:
        print(f"  -> Konnte {land} nicht komplett verarbeiten (Fehler: {e})")

# 3. Tabelle formatieren
monatsnamen = ['Jan', 'Feb', 'Mär', 'Apr', 'Mai', 'Jun', 'Jul', 'Aug', 'Sep', 'Okt', 'Nov', 'Dez']
eu_co2_emissionen.index = monatsnamen

eu_co2_emissionen = eu_co2_emissionen.loc[:, (eu_co2_emissionen != 0).any(axis=0)]

# Wir sortieren die Spalten nach ihrer CO2-Intensität, 
# damit die dreckigsten Energieträger im Balken unten und die saubersten oben liegen!
sortierte_spalten = sorted(eu_co2_emissionen.columns, key=lambda x: co2_faktoren.get(x, 0), reverse=True)
eu_co2_emissionen = eu_co2_emissionen[sortierte_spalten]

# 4. Das gestapelte Balkendiagramm zeichnen
import matplotlib.pyplot as plt

eu_co2_emissionen.plot(
    kind='bar', 
    stacked=True, 
    figsize=(14, 8), 
    colormap='tab20',   
    edgecolor='white',
    linewidth=0.5
)

plt.title("Lebenszyklus-CO2-Emissionen der europ. Stromerzeugung (2025)", fontsize=15)
plt.ylabel("Ausgestoßenes CO2-Äquivalent in Millionen Tonnen (Mt)", fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Legende ordentlich platzieren
plt.legend(title="Energieträger (IPCC Lifecycle)", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

plt.savefig(diagramm_pfad("Diagramme_2025", "co2_lebenszyklus.png"), dpi=150, bbox_inches="tight")
plt.show()